# A scholarship offer changes the comparison

**A guided investigation · Chapter 6 · 30–45 minutes · No Python experience needed**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmg777/intro2causal/blob/main/notebooks_colab/explore-synthetic_rct.ipynb)

> **The question**  
> What does randomized scholarship assignment reveal about schooling and earnings?

Suppose a scholarship is offered by lottery. Recipients may spend longer in school, but they do not all acquire the same amount of education. This teaching simulation separates the randomized offer from the schooling that follows it. Each generated person has an ability score, a scholarship assignment, years of schooling, and log earnings. Ability affects schooling and earnings, while the scholarship changes schooling without being selected on ability. We will compare ability distributions across the lottery groups, examine how their schooling differs, and then inspect earnings. That sequence helps avoid a common leap: an effect of offering a scholarship is not automatically an effect per additional year of education. Because the relationships are constructed, this is a safe place to explore the logic before encountering the complications of a real experiment.

### By the end, you will be able to…

- Follow a randomized offer through schooling and earnings, keeping assignment distinct from treatment received.
- Read and modify three interactive charts using short Python examples.
- Explain one reason a descriptive pattern alone cannot settle this case’s causal question.

### Before you see the evidence

Should scholarship recipients have systematically higher ability before receiving the offer?

Write two sentences now. You will return to them at the end.

**Your route:** meet the data → inspect three views → modify and try → ask the causal question.

## Your first few minutes in Colab · 3 minutes

You do not need prior Python experience. A **cell** is one block of text or code. Read the text, then press the **▶** beside the next code cell (or Shift+Enter). Its output appears underneath. Run cells in order: later cells use tables created earlier.

Choose **File → Save a copy in Drive** if you want to keep your changes. If a name is “not defined,” run the earlier setup cells again. **Runtime → Restart session and run all** gives you a fresh start. An error does not mean you have damaged the data: this notebook only reads it.

Keep a short learning log in a text cell. Before each chart, make a prediction. Afterward, write one observation with a number and one question that the chart cannot answer.

**Chart controls:** hover for values, click legend labels to hide/show a group, drag to zoom, and double-click to reset. A chart’s menu also offers a reset/autoscale control.

The setup installs Plotly Express if needed. `import` makes a library available; `pd` and `px` are short names for pandas (tables) and Plotly Express (charts).

In [ ]:
%pip install -q "plotly>=6,<7"

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# One consistent chart style; no plotting experience needed.
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Blues"
px.defaults.color_discrete_sequence = ["#2563eb", "#7c3aed", "#059669", "#64748b"]

## Meet the data · 5 minutes

**One row:** Simulated individual.  
**Stored size:** 2,000 rows × 4 columns.

| Column | What it records |
|---|---|
| `scholarship` | Randomized simulated scholarship offer: 1 offered, 0 not offered. |
| `ability` | Simulated pre-existing ability score (standard normal in Chapter 6). |
| `schooling` | Simulated completed years of schooling, clipped to 6–20 and rounded. |
| `earnings` | Simulated log earnings; the column name does not indicate dollars. |

This is simulated data, not an actual scholarship evaluation. earnings contains log earnings. Schooling is clipped and rounded by the generator.

`read_csv` downloads a table. We call it `df`. `head(5)` shows just five rows, not a separate sample used for the analysis. Square brackets select columns by their names.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/cmg777/intro2causal/main/data/ch6/synthetic_rct.csv"
df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df[['scholarship', 'ability', 'schooling', 'earnings']].head(5))

### Pause: what do the rows mean?

Choose one displayed row. Translate it into a sentence using the variable guide. Which values are categories or identifiers? Which are quantities? A numeric storage type does not necessarily make a code a meaningful measurement.

The next table checks types, missing values, and the number of distinct values for the key columns. `isna()` marks missing cells; `sum()` counts them. Missing is not zero. Ask which comparison would lose a row if one of its required values were missing.

In [ ]:
key_columns = ['scholarship', 'ability', 'schooling', 'earnings']
checks = pd.DataFrame({"storage_type": df[key_columns].dtypes.astype(str),
                       "missing_rows": df[key_columns].isna().sum(),
                       "distinct_values": df[key_columns].nunique()})
display(checks)
print("The charts below omit missing values required for each plotted quantity.")

### A small summary before plotting

`describe()` summarizes numeric columns: `count` counts available values, `mean` is the average, and `50%` is the median. The median splits ordered values into two halves. Averages of identifiers or arbitrary category codes are not useful descriptions, even if Python calculates them.

For category-like key variables, the second table counts values, including missing entries. Check these labels before interpreting a chart.

In [ ]:
numeric_keys = df[key_columns].select_dtypes(include="number").columns
if len(numeric_keys):
    display(df[numeric_keys].describe().round(2))
for column in key_columns:
    if df[column].nunique() <= 8:
        display(df[column].value_counts(dropna=False).rename("Rows").to_frame())

## Read the chart code · 2 minutes

`px.bar`, `px.scatter`, `px.line`, and `px.box` choose a chart type. `x` and `y` name columns for the axes; `color` distinguishes groups or values. `labels` translates column names for readers. `fig.show()` displays the result.

A table transformation and a plot answer different questions: first decide what each plotted row represents, then choose how to display it. Before running each chart, predict what you expect to see.

In [ ]:
# Display labels: run once; you do not need to memorize this mapping.
labels = {'ability': 'Simulated ability score',
 'ability_group': 'Ability group',
 'census': 'Census year',
 'earnings': 'Log earnings',
 'group': 'State group',
 'offer': 'Scholarship assignment',
 'plan': 'Assigned insurance plan',
 'schooling': 'Years of schooling',
 'state_name': 'Simulated state'}

## Visual investigation 1 · Start before the offer can matter

**Predict:** Are ability distributions similar across scholarship groups?

A box plot compares a pre-existing characteristic across the randomized groups. Randomization balances distributions in expectation, not exactly in every finite sample.

In [ ]:
offers = df.copy()
offers["offer"] = offers["scholarship"].map({0: "No scholarship", 1: "Scholarship offered"})
fig = px.box(offers, x="offer", y="ability", color="offer", points=False, labels=labels, title="Check a pre-existing characteristic")
fig.show()

**Explore the chart:** Compare both the medians and the spread. Small differences can arise by chance.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

The generator assigns offers independently of ability. That independence comes from the assignment mechanism, not from a box plot proving equality.

</details>

## Visual investigation 2 · Did the offer change schooling?

**Predict:** How do schooling distributions differ between lottery groups?

histnorm="percent" makes each group’s histogram sum to 100%. Overlaying transparent histograms helps compare the location and spread of schooling.

In [ ]:
fig = px.histogram(offers, x="schooling", color="offer", histnorm="percent", barmode="overlay", opacity=0.55, labels=labels, title="An offer changes schooling, not by the same amount for everyone")
fig.show()

**Explore the chart:** Toggle the groups and compare their centers. Notice overlapping schooling values.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

Scholarship assignment and schooling received are different variables. The distributional shift motivates a first-stage question in the chapter.

</details>

## Visual investigation 3 · Follow through to the outcome

**Predict:** Do mean log earnings differ across assignment groups?

This is a comparison by randomized offer. It is not a comparison of people who chose the same number of school years.

In [ ]:
means = offers.groupby("offer", as_index=False)["earnings"].mean()
fig = px.bar(means, x="offer", y="earnings", color="offer", labels=labels, title="The outcome comparison is by offer")
fig.show()

**Explore the chart:** Hover to read the difference on the log scale. Explain why it is not a dollar amount or a per-year schooling effect.

**Write before reading on:** Describe one pattern with its unit and comparison group. Then give one possible explanation.

<details>
<summary>Interpretation checkpoint</summary>

An offer effect can differ from an effect of education. Linking them requires assumptions about how the offer affects earnings.

</details>

## Make it yours · 8 minutes

### 1. Modify an example

Replace mean with median in the outcome comparison. Do the groups retain the same ordering?

Copy that chart cell before editing, so you can compare the original and modified outputs. Say what changed in your question as well as in your code.

### 2. Try a small investigation

Plot the average years of schooling in the two assignment groups.

Write a prediction first. The optional starter cell below is safe to leave unchanged; later cells do not depend on your answer.

In [ ]:
# Your investigation goes here.
# Describe your expected pattern before writing code.

<details>
<summary>Hint 1 · choose the right evidence</summary>

Use offers and group by offer.

</details>

<details>
<summary>Hint 2 · a Python starting point</summary>

Average schooling, then use px.bar with an accurate y-axis label.

</details>

<details>
<summary>Optional worked solution · open after trying</summary>

<pre><code>means = offers.groupby(&quot;offer&quot;, as_index=False)[&quot;schooling&quot;].mean()
fig = px.bar(means, x=&quot;offer&quot;, y=&quot;schooling&quot;, color=&quot;offer&quot;, labels=labels)
fig.show()</code></pre>

A mean difference in schooling shows that the assignment changes the treatment received on average. It is not itself the effect of schooling on earnings.

Copy the code into the starter cell to run it.

</details>

## What can you now say? · 4 minutes

Return to your initial prediction. Write a four-sentence evidence note:

1. **I expected…** State your prediction.
2. **I observed…** Use a chart, a comparison, and the correct units.
3. **This does not establish…** Name a causal claim the evidence cannot settle.
4. **To learn more, I would need…** Propose an additional measurement or a more credible comparison.

### Carry this question into the chapter

**What additional assumptions would let a randomized scholarship offer help identify the effect of schooling itself?**

[Continue with Chapter 6: The Wages of Schooling](https://cmg777.github.io/intro2causal/book/_book/notebooks_quarto/06-wages-of-schooling.html#why-not-an-rct) · [Return to the case collection](https://cmg777.github.io/intro2causal/data-dictionary.html#ch6-synthetic-rct-csv)

### Source and preparation notes

[Repository teaching simulation, code/python/ch6/generate_synthetic.py](https://github.com/cmg777/intro2causal/blob/main/code/python/ch6/generate_synthetic.py)

Generated by code/python/ch6/generate_synthetic.py using the continuing seed-2024 RNG stream after synthetic_ovb. Assignment is Bernoulli(0.5); schooling is clipped to 6–20 and rounded. Earnings and continuous ability are rounded to four decimals.

These exploratory charts are based on the committed CSV. They are not a complete replication of the original study. Keep the source study, this prepared extract, and your descriptive results distinct.